In [ ]:
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import joblib
from rdkit import Chem
from rdkit.Chem import Descriptors, AllChem

ELEMENTS = [
    'C', 'H', 'O', 'N', 'F', 'Cl', 'Br', 'I', 'S', 'P',
    'Na', 'K', 'Li', 'Ca', 'Mg', 'Zn', 'Pb', 'Cd', 'Hg',
    'Cu', 'Fe', 'Se', 'As', 'Al', 'Pt', 'Mn', 'Mo'
]
N_ELEM = len(ELEMENTS)
ELEM2ID = {e:i for i,e in enumerate(ELEMENTS)}

PAT_ELEM = re.compile(r'([A-Z][a-z]?)')
PAT_CHARGE = re.compile(r'(\[.*?[+-]\])')
PAT_COMPONENT = re.compile(r'\.')

SEL_DESC = [
    "MolWt", "TPSA", "MolLogP", "NumHDonors", "NumHAcceptors",
    "NumRotatableBonds", "NumAromaticRings", "FractionCSP3"
]
N_DESC = len(SEL_DESC)
FP_BITS = 256

def extract_hybrid_features(smi_list):
    feat_names = []
    feat_names.extend([f"atom_{e}" for e in ELEMENTS])
    feat_names += ["total_atoms", "component_num", "has_pos_ion", "has_neg_ion", "has_charge"]
    feat_names += ["double_bond", "triple_bond", "ring_num"]
    feat_names.extend(SEL_DESC)
    feat_names += [f"fp_{i}" for i in range(FP_BITS)]
    total_dim = len(feat_names)

    feat_mat = []
    rdkit_flag = []

    for smi in smi_list:
        smi_raw = str(smi).strip()
        feat = np.zeros(total_dim, dtype=np.float32)

        elem_matches = PAT_ELEM.findall(smi_raw)
        elem_cnt = {e:0 for e in ELEMENTS}
        for e in elem_matches:
            if e in elem_cnt:
                elem_cnt[e] += 1
        for idx, e in enumerate(ELEMENTS):
            feat[idx] = elem_cnt[e]

        total_atom = sum(elem_cnt.values())
        comp_num = len(PAT_COMPONENT.split(smi_raw))
        feat[N_ELEM] = total_atom
        feat[N_ELEM+1] = comp_num

        charge_matches = PAT_CHARGE.findall(smi_raw)
        has_pos = 1 if '+' in ''.join(charge_matches) else 0
        has_neg = 1 if '-' in ''.join(charge_matches) else 0
        has_chg = 1 if len(charge_matches) > 0 else 0
        feat[N_ELEM+2] = has_pos
        feat[N_ELEM+3] = has_neg
        feat[N_ELEM+4] = has_chg

        db = smi_raw.count('=')
        tb = smi_raw.count('#')
        ring = 0
        mol = Chem.MolFromSmiles(smi_raw)
        if mol is not None:
            rdkit_flag.append(True)
            for bond in mol.GetBonds():
                bt = bond.GetBondType()
                if bt == Chem.BondType.DOUBLE:
                    db += 1
                elif bt == Chem.BondType.TRIPLE:
                    tb += 1
            ring = mol.GetRingInfo().NumRings()
            desc_vals = []
            for dname in SEL_DESC:
                f = getattr(Descriptors, dname)
                v = f(mol)
                desc_vals.append(v if np.isfinite(v) else 0.0)
            fp = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=FP_BITS)
            fp_arr = np.array(fp, dtype=np.float32)
        else:
            rdkit_flag.append(False)
            desc_vals = [0.0]*N_DESC
            fp_arr = np.zeros(FP_BITS, dtype=np.float32)

        feat[N_ELEM+5] = db
        feat[N_ELEM+6] = tb
        feat[N_ELEM+7] = ring
        start_desc = N_ELEM + 8
        feat[start_desc : start_desc+N_DESC] = desc_vals
        start_fp = start_desc + N_DESC
        feat[start_fp : start_fp+FP_BITS] = fp_arr
        feat_mat.append(feat)
    return np.array(feat_mat), np.array(rdkit_flag)

class OptimizedDNN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, 64),
            nn.LayerNorm(64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1),
            nn.Tanh()
        )
    def forward(self, x):
        return self.net(x).squeeze(1)

def single_assay_predict(assay_id, smi_list, device):
    scaler = joblib.load(f"{assay_id}_hybrid_scaler.pkl")
    X_raw, rdkit_flag = extract_hybrid_features(smi_list)
    X_scaled = scaler.transform(X_raw)
    X_tensor = torch.from_numpy(X_scaled).to(device)
    feat_dim = X_scaled.shape[1]
    model = OptimizedDNN(feat_dim).to(device)
    model.load_state_dict(torch.load(f"{assay_id}_hybrid_dnn_best.pth", map_location=device))
    model.eval()
    with torch.no_grad():
        pred = model(X_tensor).cpu().numpy()
    return pred, rdkit_flag

if __name__ == "__main__":
    assay_list = [
        "er", "ar", "cyp19",
        "neuro1", "neuro2", "neuro3",
        "hep1", "hep2", "hep3",
        "dev1", "dev2", "dev3"
    ]

    pred_file = "chemical-to-predict.xlsx"
    smi_col = "SMILES1"

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"使用设备：{device}")

    df_input = pd.read_excel(pred_file)
    smi_list = df_input[smi_col].tolist()
    print(f"待预测分子总数：{len(smi_list)}")

    pred_result_dict = {}
    rdkit_parse_all = None

    for aid in assay_list:
        print(f"正在预测 {aid} ...")
        pred_vals, rdkit_flag = single_assay_predict(aid, smi_list, device)
        pred_result_dict[f"{aid}_pred_hitc"] = pred_vals
        if rdkit_parse_all is None:
            rdkit_parse_all = rdkit_flag

    df_out = pd.DataFrame()
    df_out["SMILES"] = smi_list
    df_out["rdkit_can_parse"] = rdkit_parse_all
    for col_name, arr in pred_result_dict.items():
        df_out[col_name] = arr

    save_path = "all_12_assay_pred_result.xlsx"
    df_out.to_excel(save_path, index=False)
    print(f"\n预测完成！结果已保存至：{save_path}")
    print("\n前10行预览：")
    print(df_out.head(10))

In [ ]:
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import joblib
from rdkit import Chem
from rdkit.Chem import Descriptors, AllChem
import warnings
warnings.filterwarnings("ignore")

ELEMENTS = [
    'C', 'H', 'O', 'N', 'F', 'Cl', 'Br', 'I', 'S', 'P',
    'Na', 'K', 'Li', 'Ca', 'Mg', 'Zn', 'Pb', 'Cd', 'Hg',
    'Cu', 'Fe', 'Se', 'As', 'Al', 'Pt', 'Mn', 'Mo'
]
N_ELEM = len(ELEMENTS)
ELEM2ID = {e:i for i,e in enumerate(ELEMENTS)}

PAT_ELEM = re.compile(r'([A-Z][a-z]?)')
PAT_CHARGE = re.compile(r'(\[.*?[+-]\])')
PAT_COMPONENT = re.compile(r'\.')

SEL_DESC = [
    "MolWt", "TPSA", "MolLogP", "NumHDonors", "NumHAcceptors",
    "NumRotatableBonds", "NumAromaticRings", "FractionCSP3"
]
N_DESC = len(SEL_DESC)
FP_BITS = 256

def extract_hybrid_features(smi_list):
    feat_names = []
    feat_names.extend([f"atom_{e}" for e in ELEMENTS])
    feat_names += ["total_atoms", "component_num", "has_pos_ion", "has_neg_ion", "has_charge"]
    feat_names += ["double_bond", "triple_bond", "ring_num"]
    feat_names.extend(SEL_DESC)
    feat_names += [f"fp_{i}" for i in range(FP_BITS)]
    total_dim = len(feat_names)

    feat_mat = []
    rdkit_flag = []

    for smi in smi_list:
        smi_raw = str(smi).strip()
        feat = np.zeros(total_dim, dtype=np.float32)
        elem_matches = PAT_ELEM.findall(smi_raw)
        elem_cnt = {e:0 for e in ELEMENTS}
        for e in elem_matches:
            if e in elem_cnt:
                elem_cnt[e] += 1
        for idx, e in enumerate(ELEMENTS):
            feat[idx] = elem_cnt[e]

        total_atom = sum(elem_cnt.values())
        comp_num = len(PAT_COMPONENT.split(smi_raw))
        feat[N_ELEM] = total_atom
        feat[N_ELEM+1] = comp_num

        charge_matches = PAT_CHARGE.findall(smi_raw)
        has_pos = 1 if '+' in ''.join(charge_matches) else 0
        has_neg = 1 if '-' in ''.join(charge_matches) else 0
        has_chg = 1 if len(charge_matches) > 0 else 0
        feat[N_ELEM+2] = has_pos
        feat[N_ELEM+3] = has_neg
        feat[N_ELEM+4] = has_chg

        db = smi_raw.count('=')
        tb = smi_raw.count('#')
        ring = 0
        mol = Chem.MolFromSmiles(smi_raw)
        if mol is not None:
            rdkit_flag.append(True)
            for bond in mol.GetBonds():
                bt = bond.GetBondType()
                if bt == Chem.BondType.DOUBLE:
                    db += 1
                elif bt == Chem.BondType.TRIPLE:
                    tb += 1
            ring = mol.GetRingInfo().NumRings()
            desc_vals = []
            for dname in SEL_DESC:
                f = getattr(Descriptors, dname)
                v = f(mol)
                desc_vals.append(v if np.isfinite(v) else 0.0)
            fp = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=FP_BITS)
            fp_arr = np.array(fp, dtype=np.float32)
        else:
            rdkit_flag.append(False)
            desc_vals = [0.0]*N_DESC
            fp_arr = np.zeros(FP_BITS, dtype=np.float32)

        feat[N_ELEM+5] = db
        feat[N_ELEM+6] = tb
        feat[N_ELEM+7] = ring
        start_desc = N_ELEM + 8
        feat[start_desc : start_desc+N_DESC] = desc_vals
        start_fp = start_desc + N_DESC
        feat[start_fp : start_fp+FP_BITS] = fp_arr
        feat_mat.append(feat)
    return np.array(feat_mat), np.array(rdkit_flag)

class ClassificationDNN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.feature_net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, 64),
            nn.LayerNorm(64),
            nn.ReLU(),
            nn.Dropout(0.3),
        )
        self.head = nn.Linear(64, 1)
    def forward(self, x):
        feat = self.feature_net(x)
        out = self.head(feat).squeeze(1)
        return out

def imm_model_predict(assay_id, smi_list, device):
    scaler = joblib.load(f"{assay_id}_immunotoxicity_scaler.pkl")
    X_raw, rdkit_flag = extract_hybrid_features(smi_list)
    X_scaled = scaler.transform(X_raw)
    feat_dim = X_scaled.shape[1]
    X_tensor = torch.from_numpy(X_scaled).to(device)
    model = ClassificationDNN(feat_dim).to(device)
    model.load_state_dict(torch.load(f"{assay_id}_immunotoxicity_dnn_best.pth", map_location=device))
    model.eval()
    with torch.no_grad():
        logits = model(X_tensor)
        pred_prob = torch.sigmoid(logits).cpu().numpy()
    return pred_prob, rdkit_flag

if __name__ == "__main__":
    imm_assay_list = ["imm1", "imm2", "imm3"]
    pred_file = "chemical-to-predict.xlsx"
    smi_column = "SMILES1"
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"运行设备：{device}")

    df_input = pd.read_excel(pred_file)
    smi_list = df_input[smi_column].tolist()
    print(f"待预测分子总数：{len(smi_list)}")

    pred_result = {}
    rdkit_parse_status = None

    for aid in imm_assay_list:
        print(f"正在预测 {aid} ...")
        prob_arr, flag_arr = imm_model_predict(aid, smi_list, device)
        pred_result[f"{aid}_pred_positive_prob"] = prob_arr
        if rdkit_parse_status is None:
            rdkit_parse_status = flag_arr

    df_output = pd.DataFrame()
    df_output["SMILES"] = smi_list
    df_output["rdkit_can_parse"] = rdkit_parse_status
    for col_name, prob_data in pred_result.items():
        df_output[col_name] = prob_data

    output_save_name = "immunotoxicity_3_model_smiles1_pred.xlsx"
    df_output.to_excel(output_save_name, index=False)
    print(f"\n预测完成！结果保存至：{output_save_name}")
    print("\n前10行数据预览：")
    print(df_output.head(10))